In [3]:
"""
Channel coverage diagnostic (memory-efficient, column-streaming)
---------------------------------------------------------------
Question: if we start with only channels 41-46, what fraction of anomaly events
will carry a detectable signature in that subset?

Method:
  1. Load is_anomaly once, find contiguous anomaly events.
  2. For each scored channel, load ONLY that column (as float32) via pyarrow.
     Compute per-channel nominal baseline (median, MAD) on non-anomaly rows.
     For each event, compute max |robust z| during the event.
     Mark that channel as "catching" the event if max |z| > Z_THRESH.
  3. Aggregate catch rates, report subset 41-46 vs. full-58 coverage.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sentinel.ml_logic.data import (
    load_target_channels,
    find_anomaly_segments,
)

Z_THRESH = 4.0
SUBSET   = [f"channel_{i}" for i in range(41, 47)]

REPO = Path.cwd().parent
TRAIN = REPO / "data" / "raw" / "train.parquet"
OUT_DIR = REPO / "results"
OUT_DIR.mkdir(exist_ok=True)


def read_col(col: str) -> np.ndarray:
    """Read a single column from train.parquet as float32."""
    tbl = pq.read_table(TRAIN, columns=[col])
    return tbl.column(0).to_numpy().astype(np.float32, copy=False)


def main() -> None:
    print("Reading is_anomaly column ...")
    labels = pq.read_table(TRAIN, columns=["is_anomaly"]).column(0).to_numpy().astype(np.int8)
    print(f"  rows: {len(labels):,}   anomaly rate: {labels.mean():.3%}")

    events = find_anomaly_segments(labels)
    lengths = np.array([e["length"] for e in events])
    print(f"  anomaly events: {len(events)}")
    print(f"  event length median / p90 / max: "
          f"{np.median(lengths):.0f} / {np.percentile(lengths, 90):.0f} / {lengths.max()}")

    scored = load_target_channels()
    print(f"  scored channels: {len(scored)}")

    nominal_mask = labels == 0
    catch_matrix = np.zeros((len(events), len(scored)), dtype=bool)
    max_z_per_event = np.zeros((len(events), len(scored)), dtype=np.float32)

    print(f"\nStreaming {len(scored)} channels from parquet ...")
    for ch_idx, ch in enumerate(scored):
        col = read_col(ch)

        nom = col[nominal_mask]
        med = float(np.median(nom))
        mad = float(np.median(np.abs(nom - med)))
        scale = 1.4826 * mad if mad > 0 else 1.0
        del nom

        for ev_idx, ev in enumerate(events):
            seg = col[ev["start"]: ev["end"] + 1]
            max_z = float(np.max(np.abs((seg - med) / scale)))
            max_z_per_event[ev_idx, ch_idx] = max_z
            if max_z > Z_THRESH:
                catch_matrix[ev_idx, ch_idx] = True
        del col

        if (ch_idx + 1) % 10 == 0 or ch_idx == len(scored) - 1:
            print(f"  {ch_idx + 1}/{len(scored)} channels processed")

    per_channel_catch = catch_matrix.mean(axis=0)
    ranked = sorted(zip(scored, per_channel_catch), key=lambda t: -t[1])

    subset_scored = [c for c in SUBSET if c in scored]
    subset_idx = [scored.index(c) for c in subset_scored]
    subset_cover = catch_matrix[:, subset_idx].any(axis=1).mean() if subset_idx else 0.0
    full_cover = catch_matrix.any(axis=1).mean()

    print("\n" + "=" * 72)
    print("RESULTS")
    print("=" * 72)
    print(f"Threshold: |robust z| > {Z_THRESH}")
    print(f"\nUpper bound — ANY of the {len(scored)} scored channels catches event: "
          f"{full_cover:.1%}")
    print(f"\nColleague's subset {SUBSET}")
    print(f"  scored subset: {subset_scored}")
    print(f"  event coverage — any subset channel catches: {subset_cover:.1%}")
    if full_cover > 0:
        print(f"  relative to full-58 upper bound:             "
              f"{subset_cover / full_cover:.1%}")

    print("\nTop 15 most informative scored channels:")
    for ch, rate in ranked[:15]:
        mark = "  <-- in 41-46" if ch in SUBSET else ""
        print(f"  {ch:<15s} {rate:6.1%}{mark}")

    print("\nBottom 10 scored channels (least informative):")
    for ch, rate in ranked[-10:]:
        mark = "  <-- in 41-46" if ch in SUBSET else ""
        print(f"  {ch:<15s} {rate:6.1%}{mark}")

    # How does subset 41-46 do on long vs. short events?
    lengths_arr = lengths
    subset_any = catch_matrix[:, subset_idx].any(axis=1) if subset_idx else np.zeros(len(events), bool)
    short = lengths_arr < np.percentile(lengths_arr, 33)
    mid = (lengths_arr >= np.percentile(lengths_arr, 33)) & (lengths_arr < np.percentile(lengths_arr, 66))
    long_ = lengths_arr >= np.percentile(lengths_arr, 66)
    print("\nSubset 41-46 coverage by event length bucket:")
    print(f"  short events (len < p33):        {subset_any[short].mean():.1%}  ({short.sum()} events)")
    print(f"  medium events (p33 <= len < p66): {subset_any[mid].mean():.1%}  ({mid.sum()} events)")
    print(f"  long events (len >= p66):         {subset_any[long_].mean():.1%}  ({long_.sum()} events)")

    pd.DataFrame({"channel": scored, "event_catch_rate": per_channel_catch}) \
        .sort_values("event_catch_rate", ascending=False) \
        .to_csv(OUT_DIR / "channel_coverage_per_channel.csv", index=False)
    np.savez(
        OUT_DIR / "channel_coverage_matrix.npz",
        catch_matrix=catch_matrix,
        max_z_per_event=max_z_per_event,
        channels=np.array(scored),
        event_starts=np.array([e["start"] for e in events]),
        event_ends=np.array([e["end"] for e in events]),
        event_lengths=lengths,
    )
    print(f"\nSaved: results/channel_coverage_per_channel.csv")
    print(f"Saved: results/channel_coverage_matrix.npz")


if __name__ == "__main__":
    main()

Reading is_anomaly column ...
  rows: 14,728,321   anomaly rate: 10.484%
  anomaly events: 190
  event length median / p90 / max: 602 / 30800 / 116061
  scored channels: 58

Streaming 58 channels from parquet ...
  10/58 channels processed
  20/58 channels processed
  30/58 channels processed
  40/58 channels processed
  50/58 channels processed
  58/58 channels processed

RESULTS
Threshold: |robust z| > 4.0

Upper bound — ANY of the 58 scored channels catches event: 61.6%

Colleague's subset ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']
  scored subset: ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']
  event coverage — any subset channel catches: 22.6%
  relative to full-58 upper bound:             36.8%

Top 15 most informative scored channels:
  channel_75       27.4%
  channel_41       22.6%  <-- in 41-46
  channel_42       22.6%  <-- in 41-46
  channel_43       22.6%  <-- in 41-46
  channel_44       22.1% 